# 📦 ACE-Net Master Dataset Archiver (Single Master Zip)
### Creates `baseline_features_all.zip` with Auto-Path Discovery (Case-Insensitive)

Awtomatikong hinahanap ng notebook na ito ang tamang path ng `Baseline preprocessed` kahit ano pa ang capitalization (`TRAIN`/`train`, `VAL`/`val`, `TEST`/`test`) at ini-zip ang buong dataset sa **iisang consolidated archive** (`baseline_features_all.zip`).

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os, sys

drive.mount('/content/drive')
print('✅ Google Drive mounted successfully!')

## Step 2: Auto-Detect Preprocessed Folder & Create Master Zip

In [ ]:
import os, time, shutil, subprocess
from pathlib import Path

# 1. Auto-discover the exact Google Drive preprocessed directory
CANDIDATE_ROOTS = [
    Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training/Baseline preprocessed'),
    Path('/content/drive/MyDrive/THESIS_MOTHERFILE/baseline_training/Baseline preprocessed'),
    Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training/baseline preprocessed'),
    Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline preprocessed'),
    Path('/content/drive/MyDrive/THESIS_MOTHERFILE/baseline_training'),
]

SOURCE_ROOT = None
for c in CANDIDATE_ROOTS:
    if c.exists():
        SOURCE_ROOT = c
        break

if not SOURCE_ROOT:
    # Fallback search inside THESIS_MOTHERFILE
    base_parent = Path('/content/drive/MyDrive/THESIS_MOTHERFILE')
    if base_parent.exists():
        for sub in base_parent.rglob('*preprocessed*'):
            if sub.is_dir():
                SOURCE_ROOT = sub
                break

if not SOURCE_ROOT or not SOURCE_ROOT.exists():
    raise FileNotFoundError("❌ Hindi mahanap ang Preprocessed folder sa Google Drive!")

print("=" * 75)
print(f"📁 Detected Source Preprocessed Root: {SOURCE_ROOT}")
print("   Subfolders found:", [p.name for p in SOURCE_ROOT.iterdir() if p.is_dir()])
print("=" * 75)

# Target paths
LOCAL_ZIP = Path('/content/baseline_features_all.zip')
DRIVE_DEST_DIR = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training')
DRIVE_DEST_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DRIVE_ZIP = DRIVE_DEST_DIR / 'baseline_features_all.zip'

start_time = time.time()
print("\n🚀 [1/2] Fast Zipping Entire Dataset to Colab Local NVMe SSD...")
print(f"    Target Archive : {LOCAL_ZIP}")
print("    (Please wait ~3 to 5 minutes for full 14k archive compression)...\n")

# Run native zip with wildcard inclusion directly from SOURCE_ROOT
cmd = f'cd "{SOURCE_ROOT}" && zip -q -r -1 "{LOCAL_ZIP}" *'
subprocess.run(cmd, shell=True, check=True)

if not LOCAL_ZIP.exists():
    raise RuntimeError("❌ Failed to create local zip archive.")

zip_size_mb = LOCAL_ZIP.stat().st_size / (1024**2)
zip_size_gb = zip_size_mb / 1024
elapsed_zip = time.time() - start_time
print(f"\n✅ [1/2] Local Zip Completed! Size: {zip_size_gb:.2f} GB ({zip_size_mb:.1f} MB) in {elapsed_zip/60:.2f} mins")

print(f"\n🚀 [2/2] Streaming single zip file to Google Drive ({FINAL_DRIVE_ZIP})...")
copy_start = time.time()
shutil.copy2(str(LOCAL_ZIP), str(FINAL_DRIVE_ZIP))
elapsed_copy = time.time() - copy_start
print(f"✅ [2/2] Copy to Google Drive Complete in {elapsed_copy:.1f}s!")

# Clean up local buffer to free SSD space
LOCAL_ZIP.unlink()

total_time = time.time() - start_time
print("\n" + "=" * 75)
print("       🎉 [SUCCESS] MASTER DATASET ARCHIVE CREATED! 🎉")
print("=" * 75)
print(f"⏱️ Total Time Elapsed : {total_time/60:.2f} minuto")
print(f"📦 Final Zip Size     : {zip_size_gb:.2f} GB")
print(f"📍 Saved Location     : {FINAL_DRIVE_ZIP}")
print("=" * 75)